# Étape 6 — LiDAR : guidance analytique depuis une banque fixe

**Pilote : sandwich, seed 0, luminance cible 0,7 ; SD1.5, DDIM 50, eta=0, CFG 7,5.**
Ouvrir un runtime T4, importer `diffusion-reward-guidance.zip`, puis exécuter les deux premières cellules de code.
Redémarrer seulement si l'installation le demande, puis reprendre au début.

La banque contient **8 lookaheads DDIM en 5 pas**, générés séquentiellement avec les seeds 10000–10007.
Leurs récompenses sont calculées une seule fois. Les pas guidés n'appellent ni VAE ni récompense.
On teste les identités, puis une génération à beta commun avec le pilote FULL, puis deux beta exploratoires.
Les sorties sont vides : aucune exécution T4 LiDAR n'est présumée à la livraison.

Cette version suit les équations LiDAR dérivées dans le rapport. La banque rapide approxime les échantillons de référence.
Ce n'est pas une reproduction intégrale des réglages du [papier LiDAR](https://arxiv.org/html/2602.03211v2) :
ni multiplicateur externe, ni fenêtre temporelle, ni normalisation des distances n'est ajouté.
Une guidance faible, une concentration des poids ou une baisse de récompense restent des résultats à analyser.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

candidates = [Path.cwd(), Path.cwd().parent, Path('/content/diffusion-reward-guidance')]
PROJECT_DIR = next((p for p in candidates if (p / 'guidance_lidar.py').is_file()), None)
if PROJECT_DIR is None:
    from google.colab import files
    print('Importer diffusion-reward-guidance.zip.')
    uploaded = files.upload()
    archive_name = next(name for name in uploaded if name.endswith('.zip'))
    destination = Path('/content').resolve()
    with zipfile.ZipFile(archive_name) as archive:
        for entry in archive.infolist():
            (destination / entry.filename).resolve().relative_to(destination)
        archive.extractall(destination)
    PROJECT_DIR = destination / 'diffusion-reward-guidance'
assert (PROJECT_DIR / 'guidance_lidar.py').is_file(), 'Archive LiDAR requise.'
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print('Projet :', PROJECT_DIR)

## 1. Environnement identique aux pilotes précédents

Aucune nouvelle dépendance. Luminance uniquement ; aucun chargement ImageReward.
Les fichiers Python communs peuvent être réutilisés hors Colab.

In [ ]:
import importlib.metadata
import subprocess

assert (3, 10) <= sys.version_info[:2] <= (3, 13), 'Python 3.10 à 3.13 requis.'
expected = dict(line.strip().split('==') for line in
                (PROJECT_DIR / 'requirements.txt').read_text(encoding='utf-8').splitlines()
                if line.strip() and not line.lstrip().startswith('#'))
torch_expected = {'torch': '2.8.0+cu126', 'torchvision': '0.23.0+cu126', 'torchaudio': '2.8.0+cu126'}
expected.update(torch_expected)
installed = {}
for package in expected:
    try:
        installed[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed[package] = None
needs_install = any(installed[p] != v for p, v in expected.items())
RESTART_REQUIRED = globals().get('RESTART_REQUIRED', False) or needs_install

def run_visible(command, log_name):
    log_dir = PROJECT_DIR / 'installation_logs'
    log_dir.mkdir(exist_ok=True)
    with (log_dir / log_name).open('w', encoding='utf-8') as log:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
            for line in process.stdout:
                print(line, end='', flush=True)
                log.write(line)
            return_code = process.wait()
    if return_code:
        raise RuntimeError(f'Échec {return_code} : voir les lignes ci-dessus et {log_dir / log_name}.')

if needs_install:
    if any(installed[p] != v for p, v in torch_expected.items()):
        run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--only-binary=:all:',
                     'torch==2.8.0', 'torchvision==0.23.0', 'torchaudio==2.8.0',
                     '--index-url', 'https://download.pytorch.org/whl/cu126'], 'lidar-pytorch.log')
    run_visible([sys.executable, '-m', 'pip', 'install', '--progress-bar', 'off', '--only-binary=:all:',
                 '-r', 'requirements.txt'], 'lidar-dependencies.log')
print('STOP : redémarrer la session puis reprendre au début.' if RESTART_REQUIRED
      else 'Versions conformes : poursuivre, sans redémarrage.')

In [ ]:
assert not RESTART_REQUIRED, 'Redémarrer réellement la session avant les imports.'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
import csv
import gc
import hashlib
import inspect
import json
import math
import platform
import shutil
import time
from datetime import datetime, timezone
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from baseline import (MODEL_ID, MODEL_REVISION, configure_torch, load_pipeline,
                      encode_prompt_cfg, make_initial_noise, sample_ddim, measure_call)
from rewards import decode_latents_differentiable, luminance_map, luminance_reward
from guidance_lidar import lidar_guidance, lidar_noise_prediction, sample_lidar

configure_torch()
assert torch.cuda.is_available(), 'Sélectionner un GPU T4 dans Colab.'
print(torch.cuda.get_device_name(0), torch.__version__, torch.version.cuda)
RUN_DIR = PROJECT_DIR / 'runs' / datetime.now(timezone.utc).strftime('lidar_%Y%m%d_%H%M%S_%f_UTC')
RUN_DIR.mkdir(parents=True, exist_ok=False)
results = []
algorithm_checks = {'gpu_status': 'pending', 'trajectories': {}}

def save_json(name, value):
    (RUN_DIR / name).write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding='utf-8')

run_visible([sys.executable, '-m', 'unittest', '-v', 'test_baseline', 'test_rewards', 'test_cov', 'test_full', 'test_lidar'], 'lidar-tests.log')
shutil.copy2(PROJECT_DIR / 'installation_logs' / 'lidar-tests.log', RUN_DIR / 'cpu_tests.txt')
print('Dossier résultats :', RUN_DIR)
bank_metadata = None
calibration = None

## 2. Entrées communes et configuration de la banque

Le U-Net et les états restent fp16 ; le VAE final et les récompenses sont fp32, comme COV/FULL.
Distances, softmax et sommes pondérées LiDAR sont calculés en fp64 (petite banque), puis la correction
en fp32 et le bruit corrigé au dtype DDIM. Cette précision ne supprime pas la concentration mathématique des poids.

`BANK_REUSE_DIR = None` construit une banque. Pour en réutiliser une, indiquer le dossier `bank` d'un ancien
run LiDAR : configuration et empreinte seront contrôlées, aucune récompense ne sera recalculée.
Les banques de prompts différents ne sont pas interchangeables.

In [ ]:
SEED, NUM_STEPS, CFG, TARGET = 0, 50, 7.5, 0.7
BANK_SIZE, LOOKAHEAD_STEPS = 8, 5
BANK_SEEDS = list(range(10000, 10000 + BANK_SIZE))
EVALUATION_SEEDS = [0, 1, 2, 3]
BANK_REUSE_DIR = None
assert len(set(BANK_SEEDS)) == BANK_SIZE and not set(BANK_SEEDS) & set(EVALUATION_SEEDS)
dev = (PROJECT_DIR / 'data/prompts_50_dev.txt').read_text(encoding='utf-8-sig').splitlines()
heldout = (PROJECT_DIR / 'data/prompts_50_heldout.txt').read_text(encoding='utf-8-sig').splitlines()
all_prompts = (PROJECT_DIR / 'data/generation_prompts.txt').read_text(encoding='utf-8-sig').splitlines()
assert not set(dev) & set(heldout)
PROMPT = dev[0]
assert PROMPT == 'a photo of a sandwich' and PROMPT in all_prompts
pipe = load_pipeline()
pipe.vae.to(dtype=torch.float32).eval().requires_grad_(False)
assert not pipe.vae.is_gradient_checkpointing
prompt_embeds, negative_prompt_embeds = encode_prompt_cfg(pipe, PROMPT)
initial_noise = make_initial_noise(pipe, SEED)
assert initial_noise.shape == (1, 4, 64, 64)
assert torch.equal(initial_noise, make_initial_noise(pipe, SEED))

def reward_fn(images):
    return luminance_reward(images, target=TARGET)

torch.save({'initial_noise': initial_noise.cpu(), 'prompt_embeds': prompt_embeds.cpu(),
            'negative_prompt_embeds': negative_prompt_embeds.cpu()}, RUN_DIR / 'inputs.pt')
with torch.no_grad():
    warm_latents, _ = sample_ddim(pipe, initial_noise, prompt_embeds, negative_prompt_embeds, num_steps=2)
    warm_image = decode_latents_differentiable(pipe.vae, warm_latents.float())
del warm_latents, warm_image
gc.collect()
torch.cuda.empty_cache()
pipe.scheduler.set_timesteps(NUM_STEPS, device=initial_noise.device)
source_files = ['baseline.py', 'rewards.py', 'guidance_cov.py', 'test_baseline.py',
                'test_rewards.py', 'test_cov.py', 'guidance_full.py', 'test_full.py',
                'guidance_lidar.py', 'test_lidar.py', 'requirements.txt', 'notebooks/05_lidar_t4.ipynb']
snapshot = RUN_DIR / 'source'
for name in source_files:
    (snapshot / name).parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PROJECT_DIR / name, snapshot / name)
environment = {
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION, 'prompt': PROMPT, 'seed': SEED,
    'resolution': [512, 512], 'num_steps': NUM_STEPS, 'eta': 0, 'cfg': CFG, 'target': TARGET,
    'unet_dtype': str(pipe.unet.dtype), 'vae_dtype': str(pipe.vae.dtype), 'state_dtype': str(initial_noise.dtype),
    'correction_dtype': 'torch.float32; cast to state dtype before DDIM',
    'initial_gradient_checkpointing': False, 'offload': False, 'reward': 'luminance',
    'lidar_weights_dtype': 'torch.float64', 'backward': False,
    'bank_size': BANK_SIZE, 'lookahead_steps': LOOKAHEAD_STEPS, 'bank_seeds': BANK_SEEDS,
    'scheduler': dict(pipe.scheduler.config), 'timesteps': pipe.scheduler.timesteps.cpu().tolist(),
    'gpu': torch.cuda.get_device_name(0), 'vram_mib': torch.cuda.get_device_properties(0).total_memory / 2**20,
    'python': platform.python_version(), 'torch': torch.__version__, 'cuda': torch.version.cuda,
    'deterministic_algorithms': torch.are_deterministic_algorithms_enabled(),
    'versions': {package: importlib.metadata.version(package) for package in expected},
    'source_sha256': {name: hashlib.sha256((PROJECT_DIR / name).read_bytes()).hexdigest() for name in source_files},
    'prompt_sha256': {name: hashlib.sha256((PROJECT_DIR / 'data' / name).read_bytes()).hexdigest()
                      for name in ['generation_prompts.txt', 'prompts_50_dev.txt', 'prompts_50_heldout.txt']},
    'cost_scope': 'sampler + diagnostics + CPU trace + final fp32 VAE decode; excludes disk/model load/text encoding/warmup',
}
save_json('environment.json', environment)
(RUN_DIR / 'pip_freeze.txt').write_text(subprocess.check_output(
    [sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')
print(PROMPT)
print(inspect.getsource(lidar_guidance))

## 3. BASE et générations sauvegardées

Les mesures incluent sampler, diagnostics, copies CPU des traces et décodage final fp32.
Le chargement, l'encodage du texte, la chauffe et les écritures disque sont exclus.
La construction de la banque et les sondes auront des mesures séparées.

In [ ]:
def run_generation(label, beta=None, selected_latents=None, selected_rewards=None):
    if (RUN_DIR / f'{label}_steps.json').exists():
        raise FileExistsError(f'{label} existe déjà : utiliser un nouveau RUN_DIR pour répéter.')
    method = 'BASE' if beta is None else 'LiDAR'
    steps = []

    @torch.no_grad()
    def generate_and_decode():
        arguments = dict(num_steps=NUM_STEPS, guidance_scale=CFG, return_trace=True)
        if method == 'BASE':
            latents, trace = sample_ddim(pipe, initial_noise, prompt_embeds, negative_prompt_embeds, **arguments)
        else:
            latents, trace, _ = sample_lidar(pipe, initial_noise, prompt_embeds, negative_prompt_embeds,
                bank_latents if selected_latents is None else selected_latents,
                bank_rewards if selected_rewards is None else selected_rewards,
                beta=beta, diagnostics=steps, **arguments)
        image = decode_latents_differentiable(pipe.vae, latents.float()).cpu()
        return latents.cpu(), trace, image

    started = time.perf_counter()
    try:
        (latents, trace, image), cost = measure_call(generate_and_decode)
        assert torch.isfinite(latents).all() and torch.isfinite(image).all()
        reward = reward_fn(image).item()
        luminance = luminance_map(image).mean().item()
        assert math.isfinite(reward) and math.isfinite(luminance)
    except Exception as error:
        save_json(f'{label}_failure.json', {'method': method, 'beta': beta, 'type': type(error).__name__,
                  'message': str(error), 'elapsed_seconds': time.perf_counter()-started})
        save_json(f'{label}_steps.json', steps)
        if isinstance(error, torch.cuda.OutOfMemoryError):
            (RUN_DIR / f'{label}_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
        algorithm_checks['gpu_status'] = 'failed'
        save_json('algorithm_checks.json', algorithm_checks)
        raise

    valid = method == 'BASE' or (len(steps) == NUM_STEPS and all(
        s['finite'] and s['state_detached'] and s['unet_graph_absent'] for s in steps))
    frozen = all(not p.requires_grad and p.grad is None for m in (pipe.unet, pipe.vae) for p in m.parameters())
    algorithm_checks['trajectories'][label] = dict(completed_and_finite=True, steps_valid=valid, frozen_no_grads=frozen)
    if not (valid and frozen):
        algorithm_checks['gpu_status'] = 'failed'
    save_json('algorithm_checks.json', algorithm_checks)
    row = dict(label=label, method=method, beta=0.0 if beta is None else beta,
        reward=reward, delta_reward_vs_base=reward-(results[0]['reward'] if results else reward),
        mean_luminance=luminance, absolute_error=abs(luminance-TARGET),
        bank_size=0 if method == 'BASE' else (bank_latents if selected_latents is None else selected_latents).shape[0],
        lookahead_steps=0 if method == 'BASE' else LOOKAHEAD_STEPS,
        min_ess=min((s['ess'] for s in steps), default=None),
        min_ess_reward=min((s['ess_reward'] for s in steps), default=None),
        concentrated_base_steps=sum(s['near_one_hot'] for s in steps),
        concentrated_reward_steps=sum(s['reward_near_one_hot'] for s in steps),
        zero_guidance_steps=sum(s['guidance_l2'] == 0 for s in steps),
        unchanged_noise_steps=sum(s['changed_noise_fraction'] == 0 for s in steps),
        strong_correction_steps=sum(s['strong_correction'] for s in steps),
        max_relative_correction=max((s['relative_noise_correction'] for s in steps), default=0), **cost)
    data = dict(latents=latents, trace=trace, image=image)
    torch.save(data, RUN_DIR / f'{label}_tensors.pt')
    pixels = (image[0].permute(1,2,0).numpy()*255).round().clip(0,255).astype('uint8')
    Image.fromarray(pixels).save(RUN_DIR / f'{label}.png')
    save_json(f'{label}_steps.json', steps)
    if steps:
        with (RUN_DIR / f'{label}_steps.csv').open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(steps[0]))
            writer.writeheader()
            writer.writerows(steps)
    results.append(row)
    save_json('results.json', results)
    with (RUN_DIR / 'results.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(row))
        writer.writeheader()
        writer.writerows(results)
    assert valid and frozen, 'Diagnostic algorithmique invalide : lire les fichiers sauvegardés.'
    print(json.dumps(row, indent=2))
    return data, steps

base, _ = run_generation('BASE')
display(Image.open(RUN_DIR / 'BASE.png'))

## 4. Banque indépendante : génération et scoring une seule fois

Chaque $a_i$ est le latent final du DDIM rapide existant. On sauvegarde son image, sa seed et
$r_i=R(D(a_i))$. Les latents sont conservés en fp32, sans prétendre restituer la précision perdue en fp16.
Les mesures séparent **génération**, **décodage + scoring**, et coût de construction total.
La convention du dernier pas DDIM reste celle de SD1.5, y compris pour le solveur court.

In [ ]:
algorithm_checks.update(gpu_status='bank_pending', bank_ready=False, identities_passed=False, comparisons_passed=False)
save_json('algorithm_checks.json', algorithm_checks)
BANK_DIR = RUN_DIR / 'bank'
if BANK_DIR.exists():
    raise FileExistsError('Banque déjà présente ; nouveau RUN_DIR pour recommencer, ou BANK_REUSE_DIR pour réutiliser.')
BANK_DIR.mkdir()
pipe.scheduler.set_timesteps(LOOKAHEAD_STEPS, device=initial_noise.device)
bank_config = {key: environment[key] for key in (
    'model_id','model_revision','prompt','resolution','cfg','target','reward','unet_dtype','vae_dtype','scheduler')}
bank_config['scheduler']={key:value for key,value in environment['scheduler'].items() if not key.startswith('_')}
bank_config.update(bank_size=BANK_SIZE, lookahead_steps=LOOKAHEAD_STEPS, seeds=BANK_SEEDS,
    eta=0.0, timesteps=pipe.scheduler.timesteps.cpu().tolist(),
    baseline_sha256=environment['source_sha256']['baseline.py'],
    rewards_sha256=environment['source_sha256']['rewards.py'],
    reward_preprocessing='fp32 VAE decode, clamp [0,1], fixed RGB luminance, negative squared mean error',
    versions=environment['versions'])

@torch.no_grad()
def decode_and_score(latent):
    image = decode_latents_differentiable(pipe.vae, latent.float())
    score = reward_fn(image)
    assert score.shape == (1,) and torch.isfinite(score).all() and torch.isfinite(image).all()
    return image.cpu(), score.cpu(), luminance_map(image).mean().item()

bank_rows = []
bank_started = time.perf_counter()
try:
    if BANK_REUSE_DIR is not None:
        reuse_dir = Path(BANK_REUSE_DIR).resolve()
        bank_metadata = json.loads((reuse_dir / 'bank_metadata.json').read_text(encoding='utf-8'))
        assert bank_metadata['config'] == bank_config, 'La configuration de la banque ne correspond pas.'
        assert hashlib.sha256((reuse_dir/'bank.pt').read_bytes()).hexdigest() == bank_metadata['bank_sha256']
        saved_bank = torch.load(reuse_dir/'bank.pt', map_location='cpu', weights_only=True)
        bank_latents_cpu, bank_rewards_cpu = saved_bank['latents'], saved_bank['rewards']
        for file in reuse_dir.glob('*.png'):
            shutil.copy2(file, BANK_DIR/file.name)
        shutil.copy2(reuse_dir/'bank.pt', BANK_DIR/'bank.pt')
        bank_metadata.update(reused=True, reused_from=str(reuse_dir), incremental_build_seconds=0.0)
    else:
        generated_bank, annotated_rewards = [], []
        for index, seed in enumerate(BANK_SEEDS):
            noise = make_initial_noise(pipe, seed)
            (latent, _), generation_cost = measure_call(sample_ddim, pipe, noise, prompt_embeds,
                negative_prompt_embeds, num_steps=LOOKAHEAD_STEPS, guidance_scale=CFG)
            (image, score, luminance), scoring_cost = measure_call(decode_and_score, latent)
            generated_bank.append(latent.detach().float().cpu())
            annotated_rewards.append(score.detach().double())
            pixels = (image[0].permute(1,2,0).numpy()*255).round().clip(0,255).astype('uint8')
            Image.fromarray(pixels).save(BANK_DIR/f'lookahead_{index:02d}_seed{seed}.png')
            bank_rows.append(dict(index=index, seed=seed, reward=score.item(), luminance=luminance,
                                  generation_cost=generation_cost, decode_scoring_cost=scoring_cost))
            save_json('bank/bank_progress.json', bank_rows)
            del latent, noise, image, score
        bank_latents_cpu = torch.cat(generated_bank).detach()
        bank_rewards_cpu = torch.cat(annotated_rewards).detach()
        del generated_bank, annotated_rewards
        torch.save({'latents':bank_latents_cpu, 'rewards':bank_rewards_cpu}, BANK_DIR/'bank.pt')
        generation_seconds = sum(row['generation_cost']['seconds'] for row in bank_rows)
        scoring_seconds = sum(row['decode_scoring_cost']['seconds'] for row in bank_rows)
        bank_metadata = dict(config=bank_config, rows=bank_rows, reused=False,
            generation_environment={key:environment.get(key) for key in ('gpu','vram_mib','python','torch','cuda')},
            generation_seconds=generation_seconds, decode_scoring_seconds=scoring_seconds,
            original_build_seconds=generation_seconds+scoring_seconds,
            incremental_build_seconds=generation_seconds+scoring_seconds,
            peak_allocated_mib=max(row[key]['peak_allocated_mib'] for row in bank_rows
                                   for key in ('generation_cost','decode_scoring_cost')),
            peak_reserved_mib=max(row[key]['peak_reserved_mib'] for row in bank_rows
                                  for key in ('generation_cost','decode_scoring_cost')),
            bank_sha256=hashlib.sha256((BANK_DIR/'bank.pt').read_bytes()).hexdigest())
    assert bank_latents_cpu.shape == (BANK_SIZE,*initial_noise.shape[1:])
    assert bank_rewards_cpu.shape == (BANK_SIZE,)
    assert torch.isfinite(bank_latents_cpu).all() and torch.isfinite(bank_rewards_cpu).all()
    assert not bank_latents_cpu.requires_grad and not bank_rewards_cpu.requires_grad
    bank_latents = bank_latents_cpu.to(initial_noise.device)
    bank_rewards = bank_rewards_cpu.to(initial_noise.device)
    save_json('bank/bank_metadata.json', bank_metadata)
    algorithm_checks['bank_ready'] = True
    save_json('algorithm_checks.json', algorithm_checks)
except Exception as error:
    save_json('bank_failure.json', dict(type=type(error).__name__, message=str(error), completed=bank_rows,
                                       elapsed_seconds=time.perf_counter()-bank_started))
    algorithm_checks['gpu_status'] = 'failed_during_bank'
    save_json('algorithm_checks.json', algorithm_checks)
    if isinstance(error, torch.cuda.OutOfMemoryError):
        (RUN_DIR/'bank_OOM.txt').write_text(torch.cuda.memory_summary(), encoding='utf-8')
    raise
finally:
    pipe.scheduler.set_timesteps(NUM_STEPS, device=initial_noise.device)

print('Récompenses de banque :', bank_rewards_cpu.tolist())
print('Génération (s) :', bank_metadata['generation_seconds'],
      '| Décodage + scoring (s) :', bank_metadata['decode_scoring_seconds'], '| Réutilisée :', bank_metadata['reused'])
bank_panel = Image.new('RGB', (4*256, math.ceil(BANK_SIZE/4)*280), 'white')
draw = ImageDraw.Draw(bank_panel)
for index, row in enumerate(bank_metadata['rows']):
    png = BANK_DIR/f"lookahead_{index:02d}_seed{row['seed']}.png"
    if png.exists():
        x, y = index%4*256, index//4*280
        draw.text((x+4,y+4), f"seed={row['seed']} R={row['reward']:.5f}", fill='black')
        bank_panel.paste(Image.open(png).resize((256,256)), (x,y+24))
bank_panel.save(RUN_DIR/'bank_panel.png')
display(bank_panel)

## 5. Identités sur les 50 états

Beta nul, récompense constante et banque de taille un doivent retrouver BASE exactement.
Ces tests n'exigent aucun gain de récompense. Les tableaux de poids restent enregistrés.

In [ ]:
algorithm_checks.update(gpu_status='identities_pending', identities_passed=False, comparisons_passed=False)
save_json('algorithm_checks.json', algorithm_checks)
assert algorithm_checks.get('bank_ready')
for label, beta, selected_latents, selected_rewards in [
    ('LiDAR_beta0',0.0,bank_latents,bank_rewards),
    ('LiDAR_constant',10.0,bank_latents,torch.ones_like(bank_rewards)),
    ('LiDAR_single',10.0,bank_latents[:1],bank_rewards[:1]),
]:
    data, steps = run_generation(label, beta, selected_latents, selected_rewards)
    checks = dict(states_exact=len(data['trace']) == len(base['trace']) == NUM_STEPS and all(
        torch.equal(a,b) for a,b in zip(data['trace'],base['trace'])),
        latents_exact=torch.equal(data['latents'],base['latents']), image_exact=torch.equal(data['image'],base['image']),
        guidance_zero=all(s['guidance_l2'] == 0 for s in steps))
    algorithm_checks[label] = checks
    if not all(checks.values()):
        algorithm_checks['gpu_status'] = 'failed'
    save_json('algorithm_checks.json', algorithm_checks)
    assert all(checks.values()), f'Identité {label} échouée : consulter les résultats sauvegardés.'
    del data, steps
algorithm_checks['identities_passed'] = True
save_json('algorithm_checks.json', algorithm_checks)

## 6. Référence FULL/COV et choix exploratoire des beta

Le dossier `data/full_pilot_reference` contient des **résultats antérieurs audités**. On contrôle leurs entrées
et leur BASE avant toute comparaison visuelle. Ils ne sont pas régénérés ; leurs temps appartiennent à un autre run.

Le premier beta est le beta commun du pilote FULL. Deux autres valeurs donnent une amplitude de 1 et 5
au terme $\beta(\max r_i-\min r_i)$ de la banque. Ce choix ne normalise pas les distances et ne garantit
aucun effet sur les poids : les log-vraisemblances peuvent dominer. Si toutes les récompenses sont égales,
la guidance est nulle ; le diagnostic est conservé. Sans référence disponible, la grille exploratoire est 1, 10, 100.

La grille est fixée **avant** les générations guidées. On sonde chaque beta aux trois mêmes états BASE :
indices 0, 24 et 49, sans extrapolation linéaire depuis beta=1.

In [ ]:
# Les clés exactes de la référence locale sont définies par son manifeste audité.
algorithm_checks.update(gpu_status='reference_and_probes_pending',probes_passed=False,comparisons_passed=False)
save_json('algorithm_checks.json',algorithm_checks)
REFERENCE_DIR = PROJECT_DIR/'data/full_pilot_reference'
reference = None
reference_checks = {'available': False, 'compatible': False}
if (REFERENCE_DIR/'reference.json').is_file():
    reference = json.loads((REFERENCE_DIR/'reference.json').read_text(encoding='utf-8'))
    reference_checks['available'] = True
    reference_checks['files_match'] = all(hashlib.sha256((REFERENCE_DIR/name).read_bytes()).hexdigest()==digest
                                         for name,digest in reference['files_sha256'].items())
    save_json('reference_checks.json',reference_checks)
    assert reference_checks['files_match'], 'La référence historique a changé : ne pas utiliser ces résultats.'
    stored = torch.load(REFERENCE_DIR/'reference_tensors.pt', map_location='cpu', weights_only=True)
    reference_checks['same_inputs'] = all(torch.equal(stored[key], value.detach().cpu()) for key,value in (
        ('initial_noise',initial_noise),('prompt_embeds',prompt_embeds),('negative_prompt_embeds',negative_prompt_embeds)))
    reference_checks['base_latents_exact'] = torch.equal(stored['base_latents'],base['latents'])
    reference_checks['base_image_exact'] = torch.equal(stored['base_image'],base['image'])
    reference_checks['same_config'] = all(reference['environment'][key] == environment[key] for key in
        ('model_id','model_revision','prompt','seed','resolution','num_steps','eta','cfg','target','reward','timesteps',
         'unet_dtype','vae_dtype','state_dtype','versions'))
    reference_checks['same_scheduler'] = (
        {key:value for key,value in reference['environment']['scheduler'].items() if not key.startswith('_')} ==
        {key:value for key,value in environment['scheduler'].items() if not key.startswith('_')})
    reference_checks['same_sources'] = all(reference['source_sha256'][key]==environment['source_sha256'][key]
                                           for key in ('baseline.py','rewards.py'))
    reference_checks['compatible'] = all(reference_checks[key] for key in
        ('same_inputs','base_latents_exact','base_image_exact','same_config','same_scheduler','same_sources','files_match')) and reference['audit_passed']
    del stored
save_json('reference_checks.json', reference_checks)
print('Référence FULL :', reference_checks)
if reference is not None:
    BETA_COMMON = float(reference['beta_shared'])
    span = (bank_rewards_cpu.max()-bank_rewards_cpu.min()).item()
    BETA_VALUES = list(dict.fromkeys([BETA_COMMON]+([1.0/span,5.0/span] if span>0 else [])))
else:
    BETA_COMMON, BETA_VALUES = None, [1.0,10.0,100.0]
assert all(math.isfinite(beta) and beta>0 for beta in BETA_VALUES)
save_json('beta_grid.json', dict(betas=BETA_VALUES, common_beta=BETA_COMMON,
    reward_range=(bank_rewards_cpu.max()-bank_rewards_cpu.min()).item(), fixed_before_generations=True,
    reason='historical common beta + reward-logit spans 1 and 5; fallback fixed grid without reference',
    bank_sha256=bank_metadata['bank_sha256']))

@torch.no_grad()
def probe_lidar(index, beta):
    timestep = pipe.scheduler.timesteps[index]
    state = initial_noise*pipe.scheduler.init_noise_sigma if index==0 else base['trace'][index-1].to(initial_noise.device)
    embeddings = torch.cat([negative_prompt_embeds,prompt_embeds])
    latent_input = pipe.scheduler.scale_model_input(torch.cat([state,state]),timestep)
    noise = pipe.unet(latent_input,timestep,encoder_hidden_states=embeddings,return_dict=False)[0]
    noise_u,noise_c = noise.chunk(2)
    _, report = lidar_noise_prediction(state, noise_u+CFG*(noise_c-noise_u), timestep, pipe.scheduler,
                                       bank_latents, bank_rewards,beta)
    return report

algorithm_checks.update(gpu_status='probes_pending', probes_passed=False, comparisons_passed=False)
save_json('algorithm_checks.json',algorithm_checks)
assert algorithm_checks.get('identities_passed')
if (RUN_DIR/'probes.json').exists():
    raise FileExistsError('Sondes déjà présentes : nouveau RUN_DIR pour répéter.')
probes = []
probes_started = time.perf_counter()
try:
    for beta in BETA_VALUES:
        for index in (0,NUM_STEPS//2-1,NUM_STEPS-1):
            report,cost = measure_call(probe_lidar,index,beta)
            report.update(step_index=index,cost=cost)
            probes.append(report)
            save_json('probes.json',probes)
except Exception as error:
    save_json('probe_failure.json',dict(type=type(error).__name__,message=str(error),
                                      elapsed_seconds=time.perf_counter()-probes_started))
    algorithm_checks['gpu_status']='failed_during_probes'
    save_json('algorithm_checks.json',algorithm_checks)
    if isinstance(error,torch.cuda.OutOfMemoryError):
        (RUN_DIR/'probe_OOM.txt').write_text(torch.cuda.memory_summary(),encoding='utf-8')
    raise
calibration = dict(betas=BETA_VALUES, probe_seconds=sum(p['cost']['seconds'] for p in probes),
                   max_relative_correction=max(p['relative_noise_correction'] for p in probes),
                   min_ess=min(p['ess'] for p in probes), min_ess_reward=min(p['ess_reward'] for p in probes))
save_json('calibration.json',calibration)
algorithm_checks['probes_passed']=True
save_json('algorithm_checks.json',algorithm_checks)
print(json.dumps(calibration,indent=2))

## 7. Première génération guidée complète

Le premier beta est le coefficient commun du pilote FULL lorsqu'il est disponible.
Les deux prochaines cellules séparent cette première trajectoire des autres essais.
Les NaN/OOM arrêtent le calcul et restent exportables. Une correction nulle ne déclenche pas de réglage caché.

In [ ]:
algorithm_checks.update(gpu_status='first_guided_pending',comparisons_passed=False)
save_json('algorithm_checks.json',algorithm_checks)
assert algorithm_checks.get('identities_passed') and algorithm_checks.get('probes_passed')
guided_labels=[]
data,steps=run_generation('LiDAR_1',BETA_VALUES[0])
guided_labels.append('LiDAR_1')
del data,steps
display(Image.open(RUN_DIR/'LiDAR_1.png'))

## 8. Autres beta exploratoires et verdict algorithmique

Les mêmes latents de banque et les mêmes récompenses sont réutilisés pour tous les beta.
Le verdict ne dépend ni d'une hausse de récompense ni d'une norme de guidance non nulle.

In [ ]:
algorithm_checks.update(gpu_status='comparisons_pending',comparisons_passed=False)
save_json('algorithm_checks.json',algorithm_checks)
assert guided_labels==['LiDAR_1'] and algorithm_checks.get('probes_passed')
for index,beta in enumerate(BETA_VALUES[1:],start=2):
    label=f'LiDAR_{index}'
    data,steps=run_generation(label,beta)
    guided_labels.append(label)
    del data,steps
required=['BASE','LiDAR_beta0','LiDAR_constant','LiDAR_single']+guided_labels
passed=(algorithm_checks.get('identities_passed') and algorithm_checks.get('bank_ready')
    and len(guided_labels)==len(BETA_VALUES) and all(label in algorithm_checks['trajectories'] and
        all(algorithm_checks['trajectories'][label].values()) for label in required))
algorithm_checks['comparisons_passed']=bool(passed)
algorithm_checks['gpu_status']='passed_for_this_pilot' if passed else 'failed'
algorithm_checks['scope']='algorithmic execution; reward improvement and nonzero guidance are not success criteria'
save_json('algorithm_checks.json',algorithm_checks)
assert passed,'Contrôles incomplets : lire algorithm_checks.json.'

## 9. Images, ESS et coûts

Les panneaux BASE/LiDAR montrent toutes les intensités essayées. Le panneau historique à beta commun
réutilise COV/FULL uniquement si les contrôles de référence passent ; il ne constitue pas un nouveau benchmark commun.

Pour une configuration donnée, coût isolé = construction de banque + génération guidée.
Avec $m$ images réellement produites avec cette banque, le coût moyen est
$(T_{banque}+\sum_{j=1}^m T_j)/m$. Ici le partage porte sur la grille du même prompt/seed :
ce n'est pas une mesure sur plusieurs seeds. Les contrôles et sondes sont chiffrés à part.

In [ ]:
rows={row['label']:row for row in results}
for label in guided_labels:
    panel=Image.new('RGB',(1024,572),'white')
    draw=ImageDraw.Draw(panel)
    for column,key in enumerate(('BASE',label)):
        row=rows[key]
        draw.text((column*512+8,6),f"{row['method']} beta={row['beta']:.6g} R={row['reward']:.6f}",fill='black')
        draw.text((column*512+8,24),f"L={row['mean_luminance']:.4f} | {row['seconds']:.2f}s | {row['peak_allocated_mib']/1024:.2f} GiB",fill='black')
        if key!='BASE':
            draw.text((column*512+8,42),f"ESS min={row['min_ess']:.2f} / {row['min_ess_reward']:.2f} | zero g: {row['zero_guidance_steps']}",fill='black')
        panel.paste(Image.open(RUN_DIR/f'{key}.png'),(column*512,60))
    panel.save(RUN_DIR/f'compare_{label}.png')
    display(panel)

if reference_checks['compatible']:
    reference_rows={row['label']:row for row in reference['results']}
    panel=Image.new('RGB',(2048,590),'white')
    draw=ImageDraw.Draw(panel)
    entries=[('BASE',RUN_DIR/'BASE.png',rows['BASE']),
             ('COV (run FULL)',REFERENCE_DIR/'COV_shared.png',reference_rows['COV_shared']),
             ('FULL (run FULL)',REFERENCE_DIR/'FULL_shared.png',reference_rows['FULL_shared']),
             ('LiDAR',RUN_DIR/'LiDAR_1.png',rows['LiDAR_1'])]
    for col,(title,path,row) in enumerate(entries):
        draw.text((col*512+8,6),title,fill='black')
        draw.text((col*512+8,24),f"beta={row['beta']:.6g} R={row['reward']:.6f} L={row['mean_luminance']:.4f}",fill='black')
        draw.text((col*512+8,42),f"Generation={row['seconds']:.2f}s | {row['peak_allocated_mib']/1024:.2f} GiB",fill='black')
        if title=='LiDAR':
            draw.text((col*512+8,60),f"Banque en plus : {bank_metadata['original_build_seconds']:.2f}s",fill='black')
        panel.paste(Image.open(path),(col*512,78))
    panel.save(RUN_DIR/'compare_common_beta_historical.png')
    shutil.copy2(REFERENCE_DIR/'reference.json',RUN_DIR/'historical_reference.json')
    display(panel)
else:
    print('Comparaison historique non affichée : référence absente ou entrées/BASE différentes.')

generation_seconds=sum(rows[label]['seconds'] for label in guided_labels)
bank_seconds=bank_metadata['original_build_seconds']
cost_summary=dict(bank_generation_seconds=bank_metadata['generation_seconds'],
    bank_decode_scoring_seconds=bank_metadata['decode_scoring_seconds'],bank_build_seconds=bank_seconds,
    bank_incremental_build_seconds=bank_metadata['incremental_build_seconds'],bank_reused=bank_metadata['reused'],
    bank_generation_environment=bank_metadata['generation_environment'],current_gpu=environment.get('gpu'),
    number_of_positive_generations=len(guided_labels),positive_generations_seconds=generation_seconds,
    total_bank_plus_positive_generations_seconds=bank_seconds+generation_seconds,
    amortized_seconds_per_positive_image=(bank_seconds+generation_seconds)/len(guided_labels),
    single_image_total_seconds={label:bank_seconds+rows[label]['seconds'] for label in guided_labels},
    probe_seconds=calibration['probe_seconds'],
    base_and_identity_seconds=sum(row['seconds'] for row in results if row['label'] not in guided_labels),
    full_pilot_measured_seconds=bank_metadata['incremental_build_seconds']+calibration['probe_seconds']+sum(r['seconds'] for r in results),
    peak_bank_or_generation_allocated_mib=max([bank_metadata['peak_allocated_mib']]+[r['peak_allocated_mib'] for r in results]),
    bank_peak_reserved_mib=bank_metadata['peak_reserved_mib'],
    scope='bank generation+decode/scoring and samplers+trace+final decode; excludes load/text/warmup/disk; historical timings from other run')
save_json('cost_summary.json',cost_summary)
print(json.dumps(cost_summary,indent=2))

# Résumé lisible ; les séries complètes restent dans les fichiers *_steps.json/csv.
stability_summary=[]
print('Methode     beta       timestep   ESS       ESS_reward  correction relative')
for label in guided_labels:
    series=json.loads((RUN_DIR/f'{label}_steps.json').read_text())
    for index in (0,NUM_STEPS//2-1,NUM_STEPS-1):
        step=series[index]
        row=dict(label=label,beta=rows[label]['beta'],step_index=index,timestep=step['timestep'],
                 ess=step['ess'],ess_reward=step['ess_reward'],relative_noise_correction=step['relative_noise_correction'])
        stability_summary.append(row)
        print(f"{label:11s} {row['beta']:10.4g} {row['timestep']:8d} {row['ess']:9.3f} "
              f"{row['ess_reward']:11.3f} {row['relative_noise_correction']:19.6g}")
save_json('stability_summary.json',stability_summary)

## 10. Export, même après un échec

Le ZIP conserve banque, récompenses, seeds et coûts, sources, configurations, images, traces, poids/ESS
et diagnostics par pas. Aucune amélioration de qualité ou fidélité à la distribution cible n'est supposée.
Un échec reste un échec ; la dernière cellule peut être lancée séparément pour l'analyser.

In [ ]:
save_json('algorithm_checks.json', algorithm_checks)
save_json('results.json', results)
archive_path = shutil.make_archive(str(RUN_DIR), 'zip', root_dir=RUN_DIR.parent, base_dir=RUN_DIR.name)
print('Résultats :', archive_path)
try:
    from google.colab import files
except ImportError:
    print('Northflank/local : récupérer cette archive depuis le système de fichiers.')
else:
    files.download(archive_path)